In [11]:
import polars as pl
import numpy as np
from pathlib import Path
import os


input_dir = Path("Final_Dataset_ATLAS")

features = [
    'met_recalc_pt', 'ht', 'mjj', 
    'ljet1_tau21', 'ljet1_tau32', 'ljet2_tau21', 'ljet2_tau32'
]

cr_regions = ["CR1L", "CR1L1B", "CR2L"]

print(f"Loading files from: {input_dir.resolve()}")

Loading files from: /mnt/passport/OpenData/NRAD_Proj/Final_Dataset_ATLAS


In [12]:
mc_list = []
data_list = []

for region in cr_regions:
    mc_path = input_dir / f"MC_{region}.parquet"
    data_path = input_dir / f"DATA_{region}.parquet"

    if mc_path.exists():
        df_mc = pl.read_parquet(mc_path)
        if df_mc.height > 0:

            df_mc_clean = df_mc.select(features + ["final_weight"]).with_columns([
                pl.lit(region).alias("origin_region"),
                pl.lit(0).alias("label")  # MC Target label = 0
            ])
            mc_list.append(df_mc_clean)
            
    if data_path.exists():
        df_data = pl.read_parquet(data_path)
        if df_data.height > 0:
            df_data_clean = df_data.select(features).with_columns([
                pl.lit(1.0).alias("final_weight"),
                pl.lit(region).alias("origin_region"),
                pl.lit(1).alias("label")  # Data Target label = 1
            ])
            data_list.append(df_data_clean)

df_combined_mc = pl.concat(mc_list)
df_combined_data = pl.concat(data_list)

print("\n--- COMBINATION SUMMARY ---")
print(f"Combined MC Shape:   {df_combined_mc.shape}")
print(f"Combined Data Shape: {df_combined_data.shape}")

print("\n" + "="*50)
print(" 🔍 RUNNING DATA INTEGRITY SANITY CHECKS")
print("="*50)

def run_sanity_checks(df: pl.DataFrame, name: str, is_mc: bool):
    print(f"\n[Checking {name} Dataset]")
    
    # Check 1: Nulls / NaNs
    null_counts = df.select([pl.col(c).null_count() for c in features]).sum_horizontal().item()
    nan_counts = df.select([pl.col(c).is_nan().sum() for c in features]).sum_horizontal().item()
    
    if null_counts == 0 and nan_counts == 0:
        print(" ✅ No missing values (NaNs/Nulls) detected.")
    else:
        print(f" ❌ WARNING: Detected {null_counts} Nulls and {nan_counts} NaNs in features!")

    # Check 2: Scaling Range Verification
    print(f" 📐 Scaled Target Feature Ranges:")
    for f in features:
        f_min = df[f].min()
        f_max = df[f].max()
        status = "✅" if (f_min >= -2.55 and f_max <= 2.55) else "⚠️ OUT OF BOUNDS"
        print(f"    - {f:<15} : Min = {f_min:>6.2f} | Max = {f_max:>6.2f}  {status}")


    if is_mc:
        neg_weights = df.filter(pl.col("final_weight") < 0)
        neg_count = neg_weights.height
        total_yield = df["final_weight"].sum()
        
        print(f" ⚖️ Weight Profiling:")
        print(f"    - Total MC Yield (SumW): {total_yield:.2f}")
        if neg_count > 0:
            print(f"    - ⚠️ Found {neg_count} events with negative weights!")
            print(f"      (Max negative weight value: {df['final_weight'].min():.4f})")
        else:
            print("    - ✅ No negative weights found.")


run_sanity_checks(df_combined_mc, "MASTER_MC_CR", is_mc=True)
run_sanity_checks(df_combined_data, "MASTER_DATA_CR", is_mc=False)


df_cr_final = pl.concat([df_combined_mc, df_combined_data])

print("\n" + "="*50)
print(" 🎉 CR DATASET READY FOR TRAIN/VAL/TEST SPLIT")
print("="*50)
print(f"Final Combined CR DataFrame Shape: {df_cr_final.shape}")
print(f"Total Target Label Balance -> Data (1): {df_cr_final['label'].sum()} | MC (0): {df_cr_final.height - df_cr_final['label'].sum()}")


--- COMBINATION SUMMARY ---
Combined MC Shape:   (383953, 10)
Combined Data Shape: (109865, 10)

 🔍 RUNNING DATA INTEGRITY SANITY CHECKS

[Checking MASTER_MC_CR Dataset]
 ✅ No missing values (NaNs/Nulls) detected.
 📐 Scaled Target Feature Ranges:
    - met_recalc_pt   : Min =  -2.50 | Max =   2.50  ✅
    - ht              : Min =  -2.50 | Max =   2.50  ✅
    - mjj             : Min =  -2.50 | Max =   2.50  ✅
    - ljet1_tau21     : Min =  -2.50 | Max =   2.50  ✅
    - ljet1_tau32     : Min =  -2.50 | Max =   2.49  ✅
    - ljet2_tau21     : Min =  -2.50 | Max =   2.50  ✅
    - ljet2_tau32     : Min =  -2.50 | Max =   2.50  ✅
 ⚖️ Weight Profiling:
    - Total MC Yield (SumW): 109865.00
    - ⚠️ Found 27796 events with negative weights!
      (Max negative weight value: -3.1499)

[Checking MASTER_DATA_CR Dataset]
 ✅ No missing values (NaNs/Nulls) detected.
 📐 Scaled Target Feature Ranges:
    - met_recalc_pt   : Min =  -2.50 | Max =   2.49  ✅
    - ht              : Min =  -2.50 | Max = 

In [13]:
# Check Weights

print(sum(df_combined_mc["final_weight"]))
print(len(df_combined_data))


109864.99999999999
109865


In [14]:
df_cr_final

met_recalc_pt,ht,mjj,ljet1_tau21,ljet1_tau32,ljet2_tau21,ljet2_tau32,final_weight,origin_region,label
f64,f64,f64,f64,f64,f64,f64,f64,str,i32
-1.92808,-1.555355,-2.270318,1.513103,1.709764,-1.264876,1.468204,-0.002236,"""CR1L""",0
-2.428735,-2.18053,-2.430567,0.026364,0.983081,0.372115,1.008248,0.006601,"""CR1L""",0
-2.471137,-2.221748,-2.424605,-0.066007,1.199589,-0.053801,1.432311,0.003274,"""CR1L""",0
-2.029944,-1.650237,-2.449857,0.447499,1.183097,0.415117,0.670974,0.001244,"""CR1L""",0
0.738137,-0.878298,-2.293967,1.290484,0.726982,-0.7326,1.88482,-0.002329,"""CR1L""",0
…,…,…,…,…,…,…,…,…,…
-2.226162,-2.133344,-2.317013,1.722828,1.766027,1.088128,1.048703,1.0,"""CR2L""",1
-1.667215,-1.535753,-2.086301,1.282964,1.528116,0.540572,1.153862,1.0,"""CR2L""",1
0.534414,-1.017048,-2.054415,1.135831,0.51237,0.078873,1.526254,1.0,"""CR2L""",1


In [15]:
from sklearn.model_selection import train_test_split

output_dir = input_dir / "CR_Dataset_Splits"
print(f"\nOutput directory for splits: {output_dir.resolve()}")

TEST_SIZE = 0.20
RANDOM_SEED = 2218

print("Executing Stratified Train/Test Split (80/20)...")

indices = np.arange(df_cr_final.height)
labels = df_cr_final["label"].to_numpy()

print(labels)

train_idx, test_idx = train_test_split(
    indices, 
    test_size=TEST_SIZE, 
    stratify=labels, 
    random_state=RANDOM_SEED
)

df_train_all = df_cr_final[train_idx]
df_test_all = df_cr_final[test_idx]

df_mc_train = df_train_all.filter(pl.col("label") == 0)
df_data_train = df_train_all.filter(pl.col("label") == 1)

df_mc_test = df_test_all.filter(pl.col("label") == 0)
df_data_test = df_test_all.filter(pl.col("label") == 1)

# =====================================================================
# RE-NORMALIZE MC WEIGHTS & SAVE TO NEW COLUMN
# =====================================================================

# 1. Scale Train MC
train_scale_factor = df_data_train.height / df_mc_train["final_weight"].sum()
df_mc_train = df_mc_train.with_columns(
    (pl.col("final_weight") * train_scale_factor).alias("scaled_weight")
)
# Add matching column to Data (unscaled) to keep schemas identical
df_data_train = df_data_train.with_columns(
    pl.col("final_weight").alias("scaled_weight")
)

# 2. Scale Test MC
test_scale_factor = df_data_test.height / df_mc_test["final_weight"].sum()
df_mc_test = df_mc_test.with_columns(
    (pl.col("final_weight") * test_scale_factor).alias("scaled_weight")
)
# Add matching column to Data (unscaled) to keep schemas identical
df_data_test = df_data_test.with_columns(
    pl.col("final_weight").alias("scaled_weight")
)
# =====================================================================

print("\n" + "="*85)
print(" 📊 SPLIT VERIFICATION REPORT (TRAIN & TEST ONLY)")
print("="*85)

def print_split_stats(df_mc: pl.DataFrame, df_data: pl.DataFrame, split_name: str):
    mc_count = df_mc.height
    data_count = df_data.height
    total = mc_count + data_count
    
    mc_pct = (mc_count / total) * 100 if total > 0 else 0
    data_pct = (data_count / total) * 100 if total > 0 else 0
    
    mc_orig_w = df_mc["final_weight"].sum()
    mc_scaled_w = df_mc["scaled_weight"].sum()
    
    print(f"🔹 {split_name:<5} | MC File: {mc_count:<7} ({mc_pct:.1f}%) | DATA File: {data_count:<6} ({data_pct:.1f}%)")
    print(f"          | MC Original SumW: {mc_orig_w:<12.2f} | MC Scaled SumW: {mc_scaled_w:<12.2f}")

print_split_stats(df_mc_train, df_data_train, "TRAIN")
print("-" * 85)
print_split_stats(df_mc_test, df_data_test, "TEST")
print("-" * 85)
print(f"Total Rows Saved Across All Files: {df_mc_train.height + df_data_train.height + df_mc_test.height + df_data_test.height}")

print("\nSaving finalized separate split sets to disk...")

# Define paths matching your layout
paths = {
    "MC_train": output_dir / "MC_events_train.parquet",
    "DATA_train": output_dir / "DATA_events_train.parquet",
    "MC_test": output_dir / "MC_events_test.parquet",
    "DATA_test": output_dir / "DATA_events_test.parquet",
}
os.makedirs(output_dir, exist_ok=True)

# Write files
df_mc_train.write_parquet(paths["MC_train"])
df_data_train.write_parquet(paths["DATA_train"])
df_mc_test.write_parquet(paths["MC_test"])
df_data_test.write_parquet(paths["DATA_test"])

for name, path in paths.items():
    print(f" ✅ Saved: {path.name}")

print("\n🎉 Splitting complete! Ready to be processed")


Output directory for splits: /mnt/passport/OpenData/NRAD_Proj/Final_Dataset_ATLAS/CR_Dataset_Splits
Executing Stratified Train/Test Split (80/20)...
[0 0 0 ... 1 1 1]

 📊 SPLIT VERIFICATION REPORT (TRAIN & TEST ONLY)
🔹 TRAIN | MC File: 307162  (77.8%) | DATA File: 87892  (22.2%)
          | MC Original SumW: 87884.72     | MC Scaled SumW: 87892.00    
-------------------------------------------------------------------------------------
🔹 TEST  | MC File: 76791   (77.8%) | DATA File: 21973  (22.2%)
          | MC Original SumW: 21980.28     | MC Scaled SumW: 21973.00    
-------------------------------------------------------------------------------------
Total Rows Saved Across All Files: 493818

Saving finalized separate split sets to disk...
 ✅ Saved: MC_events_train.parquet
 ✅ Saved: DATA_events_train.parquet
 ✅ Saved: MC_events_test.parquet
 ✅ Saved: DATA_events_test.parquet

🎉 Splitting complete! Ready to be processed
